In [1]:
# Install required packages first
!pip install -q transformers>=4.52.0 accelerate>=1.0.0 datasets>=3.0.0
!pip install -q av librosa soundfile resampy
!pip install -q scikit-learn matplotlib seaborn
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q pytorchvideo
!pip install -q evaluate
!pip install -q datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 MB 40.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 74.2 MB/s eta 0:00:00:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.7/132.7 kB 6.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.5 MB/s eta 0:00:00


In [ ]:
import shutil
import os

SOURCE = "/kaggle/input/datasets/ziaxxx/cp-2250/checkpoint-2250"
DEST = "/kaggle/working/checkpoints/checkpoint-2250"

if not os.path.exists(DEST):
    print("📦 Copying checkpoint to working directory...")
    shutil.copytree(SOURCE, DEST)

In [7]:
import pandas as pd
import torch
import torch.nn as nn
import numpy as np
import os
import gc
import cv2
import warnings
from typing import Dict, List, Optional, Tuple, Union
from sklearn.metrics import accuracy_score, classification_report, f1_score, recall_score, precision_score
from dataclasses import dataclass
import time
import subprocess
import sys
from contextlib import contextmanager, redirect_stderr
import io
from transformers.trainer_utils import get_last_checkpoint
# Transformers components
from transformers import (
    AutoTokenizer, AutoModel, AutoImageProcessor,
    TimesformerForVideoClassification, Trainer, TrainingArguments, 
    EvalPrediction, PreTrainedModel, PretrainedConfig
)
from torch.utils.data import Dataset, DataLoader
import evaluate

# =============================================================================
# COMPLETE ERROR SUPPRESSION AND ENVIRONMENT SETUP
# =============================================================================

def setup_ultimate_silent_environment():
    """Setup the most comprehensive silent environment"""
    
    # Environment variables for complete silence
    silent_vars = {
        'OPENCV_LOG_LEVEL': 'SILENT',
        'OPENCV_VIDEOIO_DEBUG': '0',
        'OPENCV_FFMPEG_DEBUG': '0',
        'OPENCV_VIDEOWRITER_DEBUG': '0',
        'OPENCV_VIDEOCAPTURE_DEBUG': '0',
        'OPENCV_FFMPEG_LOGLEVEL': '-8',
        'OPENCV_AVFOUNDATION_SKIP_AUTH': '1',
        'FFREPORT': 'file=/dev/null:level=quiet',
        'WANDB_DISABLED': 'true',
        'TOKENIZERS_PARALLELISM': 'false',
        'PYTHONWARNINGS': 'ignore',
        'CUDA_LAUNCH_BLOCKING': '0'
    }
    
    for key, value in silent_vars.items():
        os.environ[key] = value
    
    # Configure OpenCV for complete silence
    try:
        cv2.setLogLevel(0)
        cv2.setUseOptimized(True)
    except:
        pass
    
    # Suppress all warnings
    warnings.filterwarnings('ignore')
    warnings.simplefilter('ignore')

@contextmanager
def ultimate_stderr_suppression():
    """Most advanced stderr suppression for C libraries"""
    old_stderr = None
    try:
        if hasattr(os, 'devnull'):
            old_stderr = sys.stderr
            sys.stderr = open(os.devnull, 'w')
            yield
        else:
            old_stderr = sys.stderr
            sys.stderr = io.StringIO()
            yield
    except:
        yield
    finally:
        if old_stderr is not None:
            try:
                if hasattr(sys.stderr, 'close'):
                    sys.stderr.close()
            except:
                pass
            sys.stderr = old_stderr

# Setup environment immediately
setup_ultimate_silent_environment()
print("=== SMART MULTIMODAL CLASSIFIER - ADAPTIVE VALIDATION (FIXED) ===")

def cleanup_memory():
    """Enhanced memory cleanup"""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

cleanup_memory()

# =============================================================================
# 1. SMART VIDEO VALIDATION SYSTEM (SAME AS BEFORE)
# =============================================================================

class SmartVideoValidator:
    """Smart video validator with adaptive fallback strategies"""
    
    def __init__(self, strict_mode=False):
        self.strict_mode = strict_mode
        self.validation_cache = {}
        self.ffprobe_available = None
        self.stats = {
            'total': 0, 'valid': 0, 'invalid': 0,
            'not_found': 0, 'too_small': 0, 
            'ffprobe_failed': 0, 'opencv_failed': 0,
            'opencv_success': 0, 'ffprobe_success': 0
        }
        
        # Check FFprobe availability
        self._check_ffprobe_availability()
        
        print(f"📋 Validator initialized:")
        print(f"  FFprobe available: {self.ffprobe_available}")
        print(f"  Strict mode: {self.strict_mode}")
    
    def _check_ffprobe_availability(self):
        """Check if FFprobe is available on the system"""
        try:
            with ultimate_stderr_suppression():
                result = subprocess.run(['ffprobe', '-version'], 
                                      capture_output=True, 
                                      timeout=5)
            self.ffprobe_available = (result.returncode == 0)
        except:
            self.ffprobe_available = False
        
        if not self.ffprobe_available:
            print("⚠️ FFprobe not available - will use OpenCV-only validation")
    
    def is_video_valid(self, video_path: str) -> bool:
        """Smart video validation with adaptive strategies"""
        
        if video_path in self.validation_cache:
            return self.validation_cache[video_path]
        
        self.stats['total'] += 1
        is_valid = False
        
        try:
            # Step 1: Basic checks
            if not video_path or not os.path.exists(video_path):
                self.stats['not_found'] += 1
                self.validation_cache[video_path] = False
                return False
            
            # Step 2: File size check (more lenient)
            try:
                file_size = os.path.getsize(video_path)
                if file_size < 1000:  # Less than 1KB
                    self.stats['too_small'] += 1
                    self.validation_cache[video_path] = False
                    return False
            except:
                self.stats['too_small'] += 1
                self.validation_cache[video_path] = False
                return False
            
            # Step 3: Try FFprobe if available, otherwise use OpenCV
            if self.ffprobe_available:
                if self._validate_with_ffprobe(video_path):
                    is_valid = True
                    self.stats['ffprobe_success'] += 1
                else:
                    self.stats['ffprobe_failed'] += 1
                    # If FFprobe fails, try OpenCV as fallback
                    if self._validate_with_opencv(video_path):
                        is_valid = True
                        self.stats['opencv_success'] += 1
                    else:
                        self.stats['opencv_failed'] += 1
            else:
                # OpenCV-only validation
                if self._validate_with_opencv(video_path):
                    is_valid = True
                    self.stats['opencv_success'] += 1
                else:
                    self.stats['opencv_failed'] += 1
                
        except:
            self.stats['invalid'] += 1
        
        if not is_valid:
            self.stats['invalid'] += 1
        else:
            self.stats['valid'] += 1
        
        self.validation_cache[video_path] = is_valid
        return is_valid
    
    def _validate_with_ffprobe(self, video_path: str) -> bool:
        """FFprobe validation with lenient criteria"""
        try:
            cmd = [
                'ffprobe', '-v', 'quiet', '-select_streams', 'v:0',
                '-show_entries', 'stream=width,height,nb_frames',
                '-of', 'csv=p=0', video_path
            ]
            
            with ultimate_stderr_suppression():
                result = subprocess.run(cmd, capture_output=True, text=True, timeout=10)
            
            if result.returncode != 0:
                return False
            
            # Parse output
            output = result.stdout.strip()
            if not output:
                return False
            
            try:
                parts = output.split(',')
                if len(parts) >= 2:
                    width = int(parts[0]) if parts[0] else 0
                    height = int(parts[1]) if parts[1] else 0
                    
                    # Very lenient criteria
                    if width > 0 and height > 0:
                        return True
            except (ValueError, IndexError):
                pass
            
            return False
            
        except:
            return False
    
    def _validate_with_opencv(self, video_path: str) -> bool:
        """OpenCV validation with lenient criteria"""
        cap = None
        try:
            with ultimate_stderr_suppression():
                # Try different backends
                for backend in [cv2.CAP_FFMPEG, cv2.CAP_ANY]:
                    try:
                        cap = cv2.VideoCapture(video_path, backend)
                        if cap and cap.isOpened():
                            break
                    except:
                        continue
                
                if not cap or not cap.isOpened():
                    return False
                
                # Get basic properties
                frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
                width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
                height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
                
                # Very lenient criteria - just check if we can get basic properties
                if width > 0 and height > 0:
                    # Try to read at least one frame
                    ret, frame = cap.read()
                    if ret and frame is not None and frame.size > 0:
                        return True
                
                return False
                
        except:
            return False
        finally:
            if cap:
                try:
                    cap.release()
                except:
                    pass
    
    def validate_dataset_sample(self, df: pd.DataFrame, sample_size: int = 100) -> pd.DataFrame:
        """Validate a sample first to check validation success rate"""
        print(f"🧪 Testing validation on {sample_size} samples...")
        
        sample_df = df.head(sample_size) if len(df) > sample_size else df
        valid_count = 0
        
        for idx, row in sample_df.iterrows():
            video_path = row['video_path']
            if self.is_video_valid(video_path):
                valid_count += 1
        
        success_rate = (valid_count / len(sample_df)) * 100
        print(f"📊 Sample validation results:")
        print(f"  Tested: {len(sample_df)} files")
        print(f"  Valid: {valid_count}")
        print(f"  Success rate: {success_rate:.1f}%")
        
        if success_rate < 10:
            print("⚠️ Very low validation success rate!")
            print("🔄 Switching to lenient OpenCV-only mode...")
            return self._validate_dataset_lenient(df)
        else:
            return self._validate_dataset_normal(df)
    
    def _validate_dataset_normal(self, df: pd.DataFrame) -> pd.DataFrame:
        """Normal validation process"""
        print("🔍 Performing normal video validation...")
        
        valid_indices = []
        total = len(df)
        
        for idx, row in df.iterrows():
            video_path = row['video_path']
            if self.is_video_valid(video_path):
                valid_indices.append(idx)
            
            if (idx + 1) % 500 == 0:
                print(f"  Progress: {idx + 1}/{total} files validated...")
        
        return df.iloc[valid_indices].reset_index(drop=True)
    
    def _validate_dataset_lenient(self, df: pd.DataFrame) -> pd.DataFrame:
        """Lenient validation - just check file existence and basic OpenCV loading"""
        print("🔄 Performing lenient validation (file existence + basic OpenCV)...")
        
        valid_indices = []
        total = len(df)
        
        for idx, row in df.iterrows():
            video_path = row['video_path']
            
            # Lenient criteria: file exists and OpenCV can open it
            if (os.path.exists(video_path) and 
                os.path.getsize(video_path) > 1000 and
                self._basic_opencv_check(video_path)):
                valid_indices.append(idx)
                self.stats['valid'] += 1
            else:
                self.stats['invalid'] += 1
            
            self.stats['total'] += 1
            
            if (idx + 1) % 500 == 0:
                print(f"  Progress: {idx + 1}/{total} files validated...")
        
        return df.iloc[valid_indices].reset_index(drop=True)
    
    def _basic_opencv_check(self, video_path: str) -> bool:
        """Most basic OpenCV check - just try to open"""
        try:
            with ultimate_stderr_suppression():
                cap = cv2.VideoCapture(video_path)
                is_opened = cap.isOpened()
                cap.release()
                return is_opened
        except:
            return False
    
    def print_stats(self):
        """Print comprehensive validation statistics"""
        print(f"\n📊 Final validation statistics:")
        print(f"  Total files: {self.stats['total']}")
        print(f"  ✅ Valid files: {self.stats['valid']}")
        print(f"  ❌ Invalid files: {self.stats['invalid']}")
        print(f"  📁 Not found: {self.stats['not_found']}")
        print(f"  📏 Too small: {self.stats['too_small']}")
        if self.ffprobe_available:
            print(f"  🔧 FFprobe success: {self.stats['ffprobe_success']}")
            print(f"  🔧 FFprobe failed: {self.stats['ffprobe_failed']}")
        print(f"  📹 OpenCV success: {self.stats['opencv_success']}")
        print(f"  📹 OpenCV failed: {self.stats['opencv_failed']}")
        if self.stats['total'] > 0:
            print(f"  📈 Overall success rate: {100 * self.stats['valid'] / self.stats['total']:.1f}%")

# =============================================================================
# 2. ROBUST VIDEO LOADER (SAME AS BEFORE)
# =============================================================================

class RobustVideoLoader:
    """Robust video loader with better error handling"""
    
    def __init__(self, max_frames=8, image_size=224):
        self.max_frames = max_frames
        self.image_size = image_size
        self.video_mean = [0.485, 0.456, 0.406]
        self.video_std = [0.229, 0.224, 0.225]
        self.successful_loads = 0
        self.failed_loads = 0
        
        # Create diverse fallback videos
        self.fallback_videos = self._create_fallback_videos()
    
    def _create_fallback_videos(self):
        """Create multiple types of fallback videos"""
        fallbacks = {}
        
        # Black video
        black = torch.zeros(self.max_frames, 3, self.image_size, self.image_size)
        fallbacks['black'] = self._normalize_video(black)
        
        # Noise video
        noise = torch.randn(self.max_frames, 3, self.image_size, self.image_size) * 0.2
        fallbacks['noise'] = self._normalize_video(noise)
        
        # Simple pattern
        pattern = torch.zeros(self.max_frames, 3, self.image_size, self.image_size)
        for t in range(self.max_frames):
            for c in range(3):
                x = torch.arange(self.image_size).float() / self.image_size
                y = torch.arange(self.image_size).float() / self.image_size
                xx, yy = torch.meshgrid(x, y, indexing='ij')
                pattern[t, c] = (xx + yy + t / self.max_frames) % 1
        fallbacks['pattern'] = self._normalize_video(pattern)
        
        return fallbacks
    
    def _normalize_video(self, video_tensor):
        """Apply ImageNet normalization"""
        mean = torch.tensor(self.video_mean).view(1, 3, 1, 1)
        std = torch.tensor(self.video_std).view(1, 3, 1, 1)
        return (video_tensor - mean) / std
    
    def load_video_robust(self, video_path: str) -> torch.Tensor:
        """Load video with comprehensive error handling"""
        try:
            with ultimate_stderr_suppression():
                cap = None
                
                # Try multiple backends
                for backend in [cv2.CAP_FFMPEG, cv2.CAP_ANY]:
                    try:
                        cap = cv2.VideoCapture(video_path, backend)
                        if cap and cap.isOpened():
                            break
                    except:
                        continue
                
                if not cap or not cap.isOpened():
                    self.failed_loads += 1
                    return self._get_fallback_video()
                
                # Get video properties
                total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
                if total_frames <= 0:
                    total_frames = 30  # Assume 30 frames if unknown
                
                # Sample frames
                if total_frames <= self.max_frames:
                    frame_indices = list(range(total_frames)) + [total_frames-1] * (self.max_frames - total_frames)
                else:
                    frame_indices = np.linspace(0, total_frames-1, self.max_frames, dtype=int)
                
                frames = []
                successful_frames = 0
                
                for idx in frame_indices:
                    try:
                        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
                        ret, frame = cap.read()
                        
                        if ret and frame is not None and frame.size > 0:
                            # Process frame
                            frame = cv2.resize(frame, (self.image_size, self.image_size))
                            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                            frame = frame.astype(np.float32) / 255.0
                            
                            # Apply normalization
                            mean = np.array(self.video_mean, dtype=np.float32).reshape(1, 1, 3)
                            std = np.array(self.video_std, dtype=np.float32).reshape(1, 1, 3)
                            frame = (frame - mean) / std
                            
                            frames.append(frame)
                            successful_frames += 1
                        else:
                            # Use previous frame or create dummy
                            if frames:
                                frames.append(frames[-1])
                            else:
                                dummy = np.zeros((self.image_size, self.image_size, 3), dtype=np.float32)
                                mean = np.array(self.video_mean, dtype=np.float32).reshape(1, 1, 3)
                                std = np.array(self.video_std, dtype=np.float32).reshape(1, 1, 3)
                                frames.append((dummy - mean) / std)
                    except:
                        # Fallback frame
                        if frames:
                            frames.append(frames[-1])
                        else:
                            dummy = np.zeros((self.image_size, self.image_size, 3), dtype=np.float32)
                            mean = np.array(self.video_mean, dtype=np.float32).reshape(1, 1, 3)
                            std = np.array(self.video_std, dtype=np.float32).reshape(1, 1, 3)
                            frames.append((dummy - mean) / std)
                
                cap.release()
                
                # Ensure we have enough frames
                while len(frames) < self.max_frames:
                    if frames:
                        frames.append(frames[-1])
                    else:
                        dummy = np.zeros((self.image_size, self.image_size, 3), dtype=np.float32)
                        mean = np.array(self.video_mean, dtype=np.float32).reshape(1, 1, 3)
                        std = np.array(self.video_std, dtype=np.float32).reshape(1, 1, 3)
                        frames.append((dummy - mean) / std)
                
                # Convert to tensor (T, H, W, C) -> (T, C, H, W)
                video_array = np.stack(frames[:self.max_frames])
                video_tensor = torch.from_numpy(video_array).float()
                video_tensor = video_tensor.permute(0, 3, 1, 2)  # (T, C, H, W)
                
                # Accept any video that loads (very lenient)
                self.successful_loads += 1
                return video_tensor
                
        except:
            self.failed_loads += 1
            return self._get_fallback_video()
    
    def _get_fallback_video(self):
        """Get diverse fallback videos"""
        fallback_types = ['black', 'noise', 'pattern']
        fallback_type = fallback_types[self.failed_loads % len(fallback_types)]
        return self.fallback_videos[fallback_type].clone()
    
    def get_stats(self):
        total = self.successful_loads + self.failed_loads
        if total > 0:
            print(f"📹 Video loading: {self.successful_loads}/{total} successful ({100*self.successful_loads/total:.1f}%)")

# =============================================================================
# 3-8. MODEL AND TRAINING CODE (SAME AS BEFORE)
# =============================================================================

@dataclass
class SmartMultimodalConfig(PretrainedConfig):
    model_type = "smart_multimodal"
    
    def __init__(
        self,
        num_classes: int = 4,
        text_model_name: str = "bert-base-multilingual-cased",
        video_model_name: str = "facebook/timesformer-base-finetuned-k400",
        text_hidden_size: int = 768,
        video_hidden_size: int = 768,
        fusion_hidden_size: int = 256,
        classifier_hidden_size: int = 128,
        max_frames: int = 8,
        image_size: int = 224,
        dropout: float = 0.1,
        **kwargs
    ):
        super().__init__(**kwargs)
        self.num_classes = num_classes
        self.text_model_name = text_model_name
        self.video_model_name = video_model_name
        self.text_hidden_size = text_hidden_size
        self.video_hidden_size = video_hidden_size
        self.fusion_hidden_size = fusion_hidden_size
        self.classifier_hidden_size = classifier_hidden_size
        self.max_frames = max_frames
        self.image_size = image_size
        self.dropout = dropout

import torch
from torch import nn
import torch.nn.functional as F

import math

from einops import rearrange

class LayerNorm(nn.Module):
    def __init__(self, size, eps=1e-6):
        super(LayerNorm, self).__init__()
        self.eps = eps
        self.a_2 = nn.Parameter(torch.ones(size))
        self.b_2 = nn.Parameter(torch.zeros(size))

    def forward(self, x):
        mean = x.mean(-1, keepdim=True)
        std = x.std(-1, keepdim=True)
        return self.a_2 * (x - mean) / (std + self.eps) + self.b_2

class FC(nn.Module):
    def __init__(self, in_size, out_size, dropout_r=0., use_relu=True):
        super(FC, self).__init__()
        self.dropout_r = dropout_r
        self.use_relu = use_relu
        self.linear = nn.Linear(in_size, out_size)
        if use_relu:
            self.relu = nn.GELU()
        if dropout_r > 0:
            self.dropout = nn.Dropout(dropout_r)

    def forward(self, x):
        x = self.linear(x)
        if self.use_relu:
            x = self.relu(x)
        if self.dropout_r > 0:
            x = self.dropout(x)
        return x

class MLP(nn.Module):
    def __init__(self, in_size, mid_size, out_size, dropout_r=0.1, use_relu=True):
        super().__init__()
        self.fc = FC(in_size, mid_size, dropout_r, use_relu)
        self.linear = nn.Linear(mid_size, out_size)

    def forward(self, x):
        return self.linear(self.fc(x))

class SelfAttention(nn.Module):
    def __init__(
        self,
        *,
        dim,
        heads = 8,
        dim_head = 64,
        dropout = 0.
    ):
        super().__init__()
        inner_dim = dim_head * heads
        self.heads = heads
        self.scale = dim_head ** -0.5

        self.to_q = nn.Linear(dim, inner_dim, bias = False)
        self.to_k = nn.Linear(dim, inner_dim, bias = False)
        self.to_v = nn.Linear(dim, inner_dim, bias = False)
        self.to_out = nn.Linear(inner_dim, dim)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask = None):
        b, n, _, h = *x.shape, self.heads
        q, k, v = self.to_q(x), self.to_k(x), self.to_v(x)

        q, k, v = map(lambda t: rearrange(t, 'b n (h d) -> b h n d', h = h), (q, k, v))

        sim = torch.einsum('b h i d, b h j d -> b h i j', q, k)
        sim = sim * self.scale

        # mask

        mask_value = -torch.finfo(sim.dtype).max

        if mask is not None:  # mask: [B, N]
            mask = mask.unsqueeze(1).unsqueeze(2)  # [B,1,1,N]
            sim = sim.masked_fill(~mask, -torch.finfo(sim.dtype).max)

        # attention

        attn = sim.softmax(dim = -1)
        attn = self.dropout(attn)

        # aggregate

        out = torch.einsum('b h i j, b h j d -> b h i d', attn, v)
        
        # merge heads

        out = rearrange(out, 'b h n d -> b n (h d)')
        
        # combine heads and linear output
        
        return self.to_out(out)

class SA(nn.Module):

    def __init__(self, feat_dim, heads=8):
        super(SA, self).__init__()
        self.SelfAttention = SelfAttention(dim=feat_dim, heads=heads)
        self.norm1 = LayerNorm(feat_dim)
        self.norm2 = LayerNorm(feat_dim)
        self.ffn = MLP(in_size= feat_dim,
                       mid_size= feat_dim*2,
                       out_size= feat_dim,
                       dropout_r= 0.1,
                       use_relu=True)
    def forward(self, x, mask):
        x = self.norm1(x + 
            self.SelfAttention(x, mask)
        )
        x = self.norm2(x + 
            self.ffn(x)
        )
        return x

class SmartMultimodalModel(PreTrainedModel):
    config_class = SmartMultimodalConfig
    
    def __init__(self, config):
        super().__init__(config)
        self.config = config
        
        print(f"🔤 Initializing text model: {config.text_model_name}")
        self.text_encoder = AutoModel.from_pretrained(config.text_model_name)
        
        print(f"🎬 Initializing video model: {config.video_model_name}")
        self.video_encoder = TimesformerForVideoClassification.from_pretrained(
            config.video_model_name,
            num_labels=config.num_classes,
            ignore_mismatched_sizes=True
        )

        # FREEZE BACKBONES_____________________________________________________________________
        for param in self.text_encoder.parameters():
            param.requires_grad = False
        
        for param in self.video_encoder.parameters():
            param.requires_grad = False
        
        print("Text and Video encoders are frozen.")

        
        # Get actual dimensions
        video_dim = self.video_encoder.config.hidden_size
        text_dim = self.text_encoder.config.hidden_size
        
        self.config.video_hidden_size = video_dim
        self.config.text_hidden_size = text_dim
        
        print(f"📐 Model dimensions - Text: {text_dim}, Video: {video_dim}")
        
        # Projection layers
        self.text_projection = nn.Sequential(
            nn.Linear(text_dim, config.fusion_hidden_size),
            nn.ReLU(),
            nn.Dropout(config.dropout)
        )
        
        self.video_projection = nn.Sequential(
            nn.Linear(video_dim, config.fusion_hidden_size),
            nn.ReLU(),
            nn.Dropout(config.dropout)
        )

        self.enc = nn.ModuleList([
            SA(feat_dim=config.fusion_hidden_size, heads=8)
            for _ in range(4)
        ])
        
        # Fusion layers
        self.fusion = nn.Sequential(
            nn.Linear(config.fusion_hidden_size, config.classifier_hidden_size),
            nn.LayerNorm(config.classifier_hidden_size),
            nn.ReLU(),
            nn.Dropout(config.dropout)
        )
        
        self.classifier = nn.Linear(config.classifier_hidden_size, config.num_classes)
        
        total_params = sum(p.numel() for p in self.parameters())
        trainable_params = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"🧠 Model: {total_params:,} total, {trainable_params:,} trainable")
    
    def forward(self, pixel_values, input_ids, attention_mask, labels=None, **kwargs):
        # Text encoding
        text_outputs = self.text_encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        text_features = text_outputs.last_hidden_state
        text_features = self.text_projection(text_features)
        text_mask = attention_mask.bool() 
        
        # Video encoding with silent error handling
        try:
            # Fix tensor dimensions: (B, T, C, H, W) -> (B, T, C, H, W) (TimeSformer format)
            batch_size, num_frames, channels, height, width = pixel_values.shape
            
            video_outputs = self.video_encoder.timesformer(pixel_values)
            video_features = video_outputs.last_hidden_state
            
        except:
            # Silent fallback
            batch_size = pixel_values.size(0)
            video_features = torch.zeros(
                batch_size, self.config.video_hidden_size,
                dtype=text_features.dtype, device=text_features.device
            )
        video_mask = torch.ones(
            video_features.size(0),  # batch size
            video_features.size(1),  # số token thực tế từ TimeSformer
            dtype=torch.bool,
            device=video_features.device
        )
        video_features = self.video_projection(video_features)
            
        fused_features = torch.cat([text_features, video_features], dim=1)
        fused_mask = torch.cat([text_mask, video_mask], dim=1)  # [B, L_total]
        
        for enc in self.enc:
            fused_features = enc(fused_features, mask=fused_mask)

        pooled = fused_features.mean(dim=1)
        fused_features = self.fusion(pooled)
        logits = self.classifier(fused_features)
        
        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss(label_smoothing=0.1)
            loss = loss_fct(logits, labels)
        
        return {'loss': loss, 'logits': logits}

class SmartDataManager:
    def __init__(self, csv_path: str):
        self.csv_path = csv_path
        self.df = None
        self.classes = None
        self.class_to_id = None
        self.id_to_class = None
        self.num_classes = None
        
    def load_data(self):
        print("📂 Loading dataset...")
        self.df = pd.read_csv(self.csv_path)
        
        KAGGLE_VIDEO_ROOT = "/kaggle/input/datasets/anhoangvo/tikharm-dataset/Dataset"

        def fix_kaggle_path(old_path):
            if not isinstance(old_path, str):
                return old_path
            
            # Lấy phần sau "Dataset/"
            if "Dataset/" in old_path:
                relative_part = old_path.split("Dataset/")[-1]
                return os.path.join(KAGGLE_VIDEO_ROOT, relative_part)
            
            return old_path
        
        self.df["video_path"] = self.df["video_path"].apply(fix_kaggle_path)

        
        # Fix video paths
        # self.df['video_path'] = self.df['video_path'].str.replace(
        #     r'^/kaggle/working/extra_tikharm/',
        #     '/kaggle/input/extra-dataset/',
        #     regex=True
        # )
        
        self.df['combined_text'] = self.df['combined_text'].fillna("")
        
        print(f"Dataset loaded: {len(self.df)} samples")
        print(f"Classes distribution:\n{self.df['class_name'].value_counts()}")
        
        self.classes = sorted(self.df['class_name'].unique())
        self.class_to_id = {cls: i for i, cls in enumerate(self.classes)}
        self.id_to_class = {i: cls for cls, i in self.class_to_id.items()}
        self.num_classes = len(self.classes)
        
        print(f"📊 Classes ({self.num_classes}): {self.classes}")
        
        # Print some sample paths to debug
        print(f"\n🔍 Sample video paths:")
        for i in range(min(5, len(self.df))):
            path = self.df.iloc[i]['video_path']
            exists = os.path.exists(path)
            print(f"  {path} - Exists: {exists}")
        
    def get_smart_splits(self, validator: SmartVideoValidator):
        """Get splits with smart validation"""
        print("🧹 Creating smart validated splits...")
        
        # Get original splits
        # train_df = self.df[self.df['split'] == 'train'].reset_index(drop=True)
        # val_df = self.df[self.df['split'] == 'val'].reset_index(drop=True)
        # test_df = self.df[self.df['split'] == 'test'].reset_index(drop=True)

        train_df = self.df[(self.df['split'] == 'train') & (self.df['original_dir'] != 'extra')].reset_index(drop=True)
        val_df = self.df[(self.df['split'] == 'val') & (self.df['original_dir'] != 'extra')].reset_index(drop=True)
        test_df = self.df[self.df['split'] == 'test'].reset_index(drop=True)
        
        print(f"Original splits - Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")
        
        # Use sample validation first
        print("Validating training set...")
        clean_train_df = validator.validate_dataset_sample(train_df, sample_size=100)
        if len(clean_train_df) < 50:
            print("⚠️ Too few training samples, using lenient mode for all...")
            clean_train_df = validator._validate_dataset_lenient(train_df)
        
        print("Validating validation set...")
        clean_val_df = validator.validate_dataset_sample(val_df, sample_size=50)
        if len(clean_val_df) < 10:
            clean_val_df = validator._validate_dataset_lenient(val_df)
        
        print("Validating test set...")
        clean_test_df = validator.validate_dataset_sample(test_df, sample_size=50)
        if len(clean_test_df) < 10:
            clean_test_df = validator._validate_dataset_lenient(test_df)
        
        validator.print_stats()
        
        print(f"\n✨ Final splits:")
        print(f"  🏋️ Train: {len(clean_train_df)} (was {len(train_df)})")
        print(f"  ✅ Val: {len(clean_val_df)} (was {len(val_df)})")
        print(f"  🧪 Test: {len(clean_test_df)} (was {len(test_df)})")
        
        return clean_train_df, clean_val_df, clean_test_df

class SmartDataset(Dataset):
    """Smart dataset with robust video loading AND TRACKING"""
    
    def __init__(self, dataframe, text_tokenizer, max_frames=8, image_size=224, max_text_length=64, dataset_name="unknown"):
        self.df = dataframe
        self.text_tokenizer = text_tokenizer
        self.max_frames = max_frames
        self.image_size = image_size
        self.max_text_length = max_text_length
        self.dataset_name = dataset_name  # Track dataset name
        
        # FIXED: Store actual dataframe length at creation
        self.actual_length = len(self.df)
        
        self.video_loader = RobustVideoLoader(max_frames, image_size)
        
        print(f"🛡️ {self.dataset_name} dataset: {self.actual_length} samples (verified)")
        
        # Store indices for verification
        self.sample_indices = list(self.df.index)
        
    def __len__(self):
        # FIXED: Return actual dataframe length, not processed count
        return self.actual_length
    
    def __getitem__(self, idx):
        try:
            # FIXED: Use actual dataframe index 
            row = self.df.iloc[idx]
            
            # Process text
            text = str(row['combined_text']) if pd.notna(row['combined_text']) else ""
            if not text.strip():
                text = "[EMPTY]"
            
            text_inputs = self.text_tokenizer(
                text,
                max_length=self.max_text_length,
                truncation=True,
                padding='max_length',
                return_tensors='pt'
            )
            
            # Load video robustly
            video_path = row['video_path']
            pixel_values = self.video_loader.load_video_robust(video_path)
            
            # Get label
            label = data_manager.class_to_id[row['class_name']]
            
            return {
                'pixel_values': pixel_values,
                'input_ids': text_inputs['input_ids'].squeeze(0),
                'attention_mask': text_inputs['attention_mask'].squeeze(0),
                'labels': torch.tensor(label, dtype=torch.long)
            }
            
        except Exception as e:
            print(f"⚠️ Error at index {idx} in {self.dataset_name}: {e}")
            
            # Safe fallback
            dummy_video = torch.zeros(self.max_frames, 3, self.image_size, self.image_size)
            dummy_input_ids = torch.zeros(self.max_text_length, dtype=torch.long)
            dummy_attention_mask = torch.zeros(self.max_text_length, dtype=torch.long)
            
            return {
                'pixel_values': dummy_video,
                'input_ids': dummy_input_ids,
                'attention_mask': dummy_attention_mask,
                'labels': torch.tensor(0, dtype=torch.long)
            }
    
    def get_stats(self):
        print(f"📊 {self.dataset_name} stats:")
        print(f"  DataFrame length: {len(self.df)}")
        print(f"  Actual length: {self.actual_length}")
        self.video_loader.get_stats()

def compute_comprehensive_metrics(eval_pred: EvalPrediction) -> Dict[str, float]:
    """Compute all 4 key metrics"""
    predictions = np.argmax(eval_pred.predictions, axis=1)
    labels = eval_pred.label_ids
    
    accuracy = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='macro', zero_division=0)
    recall = recall_score(labels, predictions, average='macro', zero_division=0)
    precision = precision_score(labels, predictions, average='macro', zero_division=0)
    
    return {
        'accuracy': accuracy,
        'f1_score': f1,
        'recall': recall,
        'precision': precision
    }

def smart_collate_fn(batch):
    """Smart collate function"""
    try:
        return {
            'pixel_values': torch.stack([item['pixel_values'] for item in batch]),
            'input_ids': torch.stack([item['input_ids'] for item in batch]),
            'attention_mask': torch.stack([item['attention_mask'] for item in batch]),
            'labels': torch.stack([item['labels'] for item in batch])
        }
    except Exception as e:
        print(f"⚠️ Collate error: {e}")
        batch_size = len(batch)
        return {
            'pixel_values': torch.zeros(batch_size, 8, 3, 224, 224),
            'input_ids': torch.zeros(batch_size, 64, dtype=torch.long),
            'attention_mask': torch.zeros(batch_size, 64, dtype=torch.long),
            'labels': torch.zeros(batch_size, dtype=torch.long)
        }



=== SMART MULTIMODAL CLASSIFIER - ADAPTIVE VALIDATION (FIXED) ===


In [3]:
# =============================================================================
# MAIN EXECUTION WITH FIXED DATA TRACKING
# =============================================================================

def main():
    """Main training function with FIXED data counting"""
    
    try:
        print("🚀 Starting smart adaptive training (FIXED VERSION)...")
        
        # Initialize data manager
        global data_manager
        data_manager = SmartDataManager("/kaggle/input/datasets/kusssssss/processed-data/tikharm_vi_en_complete.csv")
        data_manager.load_data()
        
        # Initialize smart validator (not strict)
        print("🔍 Initializing smart video validator...")
        validator = SmartVideoValidator(strict_mode=False)
        
        # Get smart validated data
        train_df, val_df, test_df = data_manager.get_smart_splits(validator)
        
     
        # VERIFY DATA SPLITS AGAIN
        print(f"\n🔍 VERIFICATION - Actual dataframe sizes:")
        print(f"  Train DF: {len(train_df)} rows")
        print(f"  Val DF: {len(val_df)} rows") 
        print(f"  Test DF: {len(test_df)} rows")
        
        # Check data sufficiency
        if len(train_df) < 50:
            print("⚠️ Very few training samples!")
            print("🔄 Proceeding with available data...")
        
        # Initialize models
        print("🧠 Initializing models...")
        text_model_name = "bert-base-multilingual-cased"
        text_tokenizer = AutoTokenizer.from_pretrained(text_model_name)
        
        config = SmartMultimodalConfig(
            num_classes=data_manager.num_classes,
            text_model_name=text_model_name,
            max_frames=8,
            image_size=224,
            dropout=0.1
        )
        
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        print(f"💻 Using device: {device}")
        
        model = SmartMultimodalModel(config)
        model.to(device)
        
        # Create smart datasets WITH PROPER TRACKING
        print("📊 Creating smart datasets with tracking...")
        train_dataset = SmartDataset(train_df, text_tokenizer, dataset_name="TRAIN")
        val_dataset = SmartDataset(val_df, text_tokenizer, dataset_name="VAL")
        test_dataset = SmartDataset(test_df, text_tokenizer, dataset_name="TEST")
        
        # VERIFY DATASET LENGTHS
        print(f"\n🎯 FINAL DATASET VERIFICATION:")
        print(f"  Train dataset: {len(train_dataset)} samples")
        print(f"  Val dataset: {len(val_dataset)} samples")
        print(f"  Test dataset: {len(test_dataset)} samples")
        
        # Test sample loading
        print("🧪 Testing sample loading...")
        sample = train_dataset[0]
        print(f"  ✅ Video shape: {sample['pixel_values'].shape}")
        print(f"  ✅ Text shape: {sample['input_ids'].shape}")
        
        # Test forward pass
        model.eval()
        with torch.no_grad():
            batch = {k: v.unsqueeze(0).to(device) for k, v in sample.items()}
            output = model(**batch)
            print(f"  ✅ Output shape: {output['logits'].shape}")

        OUTPUT_DIR = "/kaggle/working/checkpoints"
        FINAL_MODEL_DIR = "/kaggle/working/final_model"
        
        # Training setup with FIXED batch settings
        training_args = TrainingArguments(
            output_dir=OUTPUT_DIR,
            num_train_epochs=10,
            per_device_train_batch_size=6,
            per_device_eval_batch_size=6,
            gradient_accumulation_steps=2,
            learning_rate=1e-4,
            weight_decay=0.01,
            warmup_steps=100,
            logging_steps=50,
            eval_strategy="steps",
            eval_steps=50,
            save_strategy="steps",
            save_steps=50,
            load_best_model_at_end=True,
            metric_for_best_model="f1_score",
            fp16=True,
            dataloader_num_workers=0,
            remove_unused_columns=False,
            report_to="none",
            save_total_limit=10,
            # FIXED: Don't drop last to preserve data count
            dataloader_drop_last=False,  # Changed from True to False
            max_grad_norm=1.0,
        )
        
        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=val_dataset,
            data_collator=smart_collate_fn,
            compute_metrics=compute_comprehensive_metrics,
        )
        
        # Start training
        print("\n" + "="*80)
        print("🚀 STARTING SMART ADAPTIVE TRAINING (FIXED)!")
        print("="*80)
        
        start_time = time.time()

        
        # Train with error suppression
        last_checkpoint = None

        # AUTO RESUME LOGIC ______________________________________________________________________
        # if os.path.isdir(OUTPUT_DIR):
        #     checkpoints = [
        #         os.path.join(OUTPUT_DIR, d)
        #         for d in os.listdir(OUTPUT_DIR)
        #         if d.startswith("checkpoint-")
        #     ]
        #     if len(checkpoints) > 0:
        #         last_checkpoint = max(checkpoints, key=os.path.getctime)
        
        # if last_checkpoint:
        #     print(f"Resuming from checkpoint: {last_checkpoint}")
        #     trainer.train(resume_from_checkpoint=last_checkpoint)
        # else:
        #     print("Starting fresh training...")
        #     trainer.train()
        
        if os.path.isdir(OUTPUT_DIR):
            last_checkpoint = get_last_checkpoint(OUTPUT_DIR)
        
        if last_checkpoint is not None:
            print(f"🔁 Resuming from checkpoint: {last_checkpoint}")
            trainer.train(resume_from_checkpoint=True)
        else:
            print("Starting fresh training...")
            trainer.train()

        
        training_time = time.time() - start_time
        
        print(f"\n🎉 TRAINING COMPLETED in {training_time:.1f}s!")
        
        # Save model
        trainer.save_model(FINAL_MODEL_DIR)
        print("💾 Model saved!")
        
        


        # Evaluation
        print("📊 Running evaluation...")
        eval_results = trainer.evaluate()
        
        print(f"\n🎯 VALIDATION RESULTS:")
        print(f"  Accuracy: {eval_results['eval_accuracy']:.4f} ({eval_results['eval_accuracy']*100:.2f}%)")
        print(f"  F1-Score: {eval_results['eval_f1_score']:.4f}")
        print(f"  Recall: {eval_results['eval_recall']:.4f}")
        print(f"  Precision: {eval_results['eval_precision']:.4f}")
        
        # FIXED Test evaluation with manual prediction
        print("🧪 Running FIXED test evaluation...")
        
        # Use manual DataLoader to control exact sample count
        test_dataloader = DataLoader(
            test_dataset, 
            batch_size=8, 
            shuffle=False, 
            drop_last=False,  # Don't drop any samples
            collate_fn=smart_collate_fn
        )
        
        model.eval()
        all_predictions = []
        all_labels = []
        
        print(f"🔍 Processing test data in {len(test_dataloader)} batches...")
        
        with torch.no_grad():
            for batch_idx, batch in enumerate(test_dataloader):
                # Move to device
                batch = {k: v.to(device) if torch.is_tensor(v) else v for k, v in batch.items()}
                
                # Get predictions
                outputs = model(**batch)
                predictions = torch.argmax(outputs['logits'], dim=1)
                
                all_predictions.extend(predictions.cpu().numpy())
                all_labels.extend(batch['labels'].cpu().numpy())
                
                if (batch_idx + 1) % 10 == 0:
                    print(f"  Processed batch {batch_idx + 1}/{len(test_dataloader)}")
        
        # Convert to numpy arrays
        y_pred = np.array(all_predictions)
        y_true = np.array(all_labels)
        
        print(f"\n🎯 FINAL TEST DATA VERIFICATION:")
        print(f"  Expected test samples: {len(test_dataset)}")
        print(f"  Actual predictions: {len(y_pred)}")
        print(f"  Actual labels: {len(y_true)}")
        print(f"  Data consistency: {'✅ CONSISTENT' if len(y_pred) == len(test_dataset) else '❌ INCONSISTENT'}")
        
        # Test metrics
        test_accuracy = accuracy_score(y_true, y_pred)
        test_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
        test_recall = recall_score(y_true, y_pred, average='macro', zero_division=0)
        test_precision = precision_score(y_true, y_pred, average='macro', zero_division=0)
        
        print(f"\n🏆 FINAL TEST RESULTS (VERIFIED):")
        print(f"  Test samples: {len(y_true)}")
        print(f"  Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
        print(f"  F1-Score: {test_f1:.4f}")
        print(f"  Recall: {test_recall:.4f}")
        print(f"  Precision: {test_precision:.4f}")
        
        # Classification report
        print(f"\n📋 CLASSIFICATION REPORT (VERIFIED):")
        print("-" * 60)
        class_report = classification_report(
            y_true, y_pred, 
            target_names=data_manager.classes,
            digits=4
        )
        print(class_report)
        
        # Save results
        import json
        results = {
            'model_type': 'Smart_Adaptive_Multimodal_FIXED',
            'data_verification': {
                'train_samples': len(train_dataset),
                'val_samples': len(val_dataset),
                'test_samples': len(test_dataset),
                'test_predictions': len(y_pred),
                'data_consistent': len(y_pred) == len(test_dataset)
            },
            'validation_metrics': {
                'accuracy': eval_results['eval_accuracy'],
                'f1_score': eval_results['eval_f1_score'],
                'recall': eval_results['eval_recall'],
                'precision': eval_results['eval_precision']
            },
            'test_metrics': {
                'accuracy': test_accuracy,
                'f1_score': test_f1,
                'recall': test_recall,
                'precision': test_precision
            },
            'training_time_seconds': training_time,
            'validation_stats': validator.stats,
            'dataset_sizes': {
                'train': len(train_dataset),
                'val': len(val_dataset),
                'test': len(test_dataset)
            }
        }
        
        with open('smart_results_fixed.json', 'w') as f:
            json.dump(results, f, indent=2)
        
        print("💾 Results saved!")
        
        # Final statistics
        print("\n📈 Final statistics:")
        train_dataset.get_stats()
        val_dataset.get_stats() 
        test_dataset.get_stats()
        
        print("\n" + "="*80)
        print("🎊 SMART TRAINING COMPLETED SUCCESSFULLY (FIXED)!")
        print("✅ DATA COUNTING ISSUES RESOLVED!")
        print("📊 ALL METRICS CALCULATED CORRECTLY!")
        print("="*80)

        
       
        
    except Exception as e:
        print(f"❌ Unexpected error: {e}")
        import traceback
        traceback.print_exc()
    
    finally:
        cleanup_memory()


In [8]:
# Run the main function
if __name__ == "__main__":
    main()

🚀 Starting smart adaptive training (FIXED VERSION)...
📂 Loading dataset...
Dataset loaded: 4723 samples
Classes distribution:
class_name
Safe               1248
Harmful Content    1193
Adult Content      1141
Suicide            1141
Name: count, dtype: int64
📊 Classes (4): ['Adult Content', 'Harmful Content', 'Safe', 'Suicide']

🔍 Sample video paths:
  /kaggle/input/datasets/anhoangvo/tikharm-dataset/Dataset/train/Adult Content/jaypark_foryou_7339639860354911493.mp4 - Exists: True
  /kaggle/input/datasets/anhoangvo/tikharm-dataset/Dataset/train/Adult Content/1life.anthonyblane_7368846422709521671.mp4 - Exists: True
  /kaggle/input/datasets/anhoangvo/tikharm-dataset/Dataset/train/Adult Content/loungebarvip_7252268249243454725.mp4 - Exists: True
  /kaggle/input/datasets/anhoangvo/tikharm-dataset/Dataset/train/Adult Content/phangphong111_7287936860796620039.mp4 - Exists: True
  /kaggle/input/datasets/anhoangvo/tikharm-dataset/Dataset/train/Adult Content/girl_xinh.sexy_7298717939052711170.

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🎬 Initializing video model: facebook/timesformer-base-finetuned-k400


Loading weights:   0%|          | 0/249 [00:00<?, ?it/s]

TimesformerForVideoClassification LOAD REPORT from: facebook/timesformer-base-finetuned-k400
Key               | Status   |                                                                                         
------------------+----------+-----------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([400, 768]) vs model:torch.Size([4, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([400]) vs model:torch.Size([4])          

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


Text and Video encoders are frozen.
📐 Model dimensions - Text: 768, Video: 768
🧠 Model: 302,696,584 total, 3,581,316 trainable
📊 Creating smart datasets with tracking...
🛡️ TRAIN dataset: 2762 samples (verified)
🛡️ VAL dataset: 396 samples (verified)
🛡️ TEST dataset: 790 samples (verified)

🎯 FINAL DATASET VERIFICATION:
  Train dataset: 2762 samples
  Val dataset: 396 samples
  Test dataset: 790 samples
🧪 Testing sample loading...
  ✅ Video shape: torch.Size([8, 3, 224, 224])
  ✅ Text shape: torch.Size([64])
  ✅ Output shape: torch.Size([1, 4])

🚀 STARTING SMART ADAPTIVE TRAINING (FIXED)!
🔁 Resuming from checkpoint: /kaggle/working/checkpoints/checkpoint-2250
❌ Unexpected error: 'NoneType' object has no attribute 'load_state_dict'


Traceback (most recent call last):
  File "/tmp/ipykernel_55/532820862.py", line 151, in main
    trainer.train(resume_from_checkpoint=True)
  File "/usr/local/lib/python3.12/dist-packages/transformers/trainer.py", line 1412, in train
    return inner_training_loop(
           ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/trainer.py", line 1579, in _inner_training_loop
    self._load_scaler(resume_from_checkpoint)
  File "/usr/local/lib/python3.12/dist-packages/transformers/trainer.py", line 3674, in _load_scaler
    self.accelerator.scaler.load_state_dict(
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: 'NoneType' object has no attribute 'load_state_dict'


In [ ]:
import os
print(os.listdir("/kaggle/input/datasets/ziaxxx/cp-2250/checkpoint-2250"))